# Jour 2 — Fichiers, base de données et premiers pas avec pandas

*Formation Python pour l'analyse de données — sujet*

**Paliers** : ★ indispensable — ★★ à faire si le ★ est fini — ★★★ pour aller plus loin.
Faites les ★ de tous les exercices avant d'attaquer les ★★★.

Réflexes sans assistant IA : `Tab` (autocomplétion), `Shift+Tab` (signature), `objet?` (aide),
`help(objet)`, `dir(objet)`, la doc officielle (https://docs.python.org/fr/3/, https://pandas.pydata.org/docs/)
et les aide-mémoire fournis.

## Les données

Le dossier `donnees/` contient un export fictif (mais réaliste) de trois tables :
`clients.csv`, `contrats.csv`, `sinistres.csv` et la même chose dans la base `assurance.db`
(SQLite, qui joue ici le rôle de Vertica). **Elles contiennent volontairement des anomalies.**

In [ ]:
from pathlib import Path

DONNEES = Path("donnees")
for fichier in sorted(DONNEES.iterdir()):
    print(f"{fichier.name:20} {fichier.stat().st_size / 1024:8.1f} Ko")

### Exercice 1 — Lire un fichier texte  ★☆☆

Affichez les 3 premières lignes de `contrats.csv` avec `open` (pas pandas).
La cellule fournie plante : lisez l'erreur. Le fichier vient d'un export Windows / SAS :
quel encodage faut-il préciser ? Affichez ensuite le nombre total de lignes.

In [ ]:
with open(DONNEES / "contrats.csv", encoding="utf-8") as f:
    for i, ligne in enumerate(f):
        print(ligne.rstrip())

### Exercice 2 — Charger les CSV avec pandas  ★☆☆

Chargez les trois fichiers dans `clients`, `contrats`, `sinistres` avec `pd.read_csv`.
Paramètres à trouver dans la doc (`pd.read_csv?`) : séparateur, séparateur décimal,
encodage, colonnes de dates au format jour/mois/année.
Le `code_postal` doit rester du texte (pourquoi ?). Vérifiez avec `.dtypes` et `.head()`.

In [ ]:
import pandas as pd

# À vous
clients = ...

**Pourquoi `dtype={"code_postal": "string"}` ?** Sinon pandas lit `01000` comme le nombre `1000`.
Ce piège est déjà arrivé dans l'export : regardez l'exercice suivant.

### Exercice 3 — Expressions régulières : contrôles de format  ★★☆

Avec le module `re`, puis avec les méthodes `.str` de pandas :
1. écrivez le motif `MOTIF_CONTRAT` du format `PPP-AAAA-NNNNNN` et testez-le avec `re.fullmatch` ;
2. comptez les numéros de contrat invalides dans `contrats` (`.str.fullmatch`) ;
3. comptez les emails invalides (un motif simple suffit : `texte@texte.texte`) ;
4. ★★★ réparez les codes postaux à 4 chiffres (`.str.zfill`).

In [ ]:
import re

MOTIF_CONTRAT = r""   # À vous
print(re.fullmatch(MOTIF_CONTRAT, "AUT-2023-000123"))

## Accès à la base de données

Ici on utilise SQLite (`assurance.db`) pour pouvoir travailler partout.
**Au datalab, seule la connexion change** ; les requêtes et `pd.read_sql` restent identiques.

```python
# Au datalab, avec verticapy (vérifié avec verticapy 1.1 / Vertica 25.4)
import verticapy as vp
vp.connect("nom_de_la_connexion")            # connexion déjà déclarée au datalab
contrats_v = vp.vDataFrame("schema.contrats")   # paresseux : rien n'est rapatrié
par_produit = contrats_v.groupby("produit", ["COUNT(*) AS nb"])   # agrégats écrits en SQL
df = par_produit.to_pandas()                 # la requête part ici ; le résultat arrive dans pandas

# ou avec le pilote vertica_python + pandas (paramètres nommés :nom)
import vertica_python
with vertica_python.connect(**infos_connexion) as cnx:   # infos lues dans l'environnement
    df = pd.read_sql("SELECT * FROM schema.contrats WHERE produit = :produit", cnx,
                     params={"produit": "AUTO"})
```

Règle pratique : **filtrer et agréger côté serveur** (SQL / verticapy), ne rapatrier dans pandas
que ce qui tient en mémoire et ce qu'il faut retravailler finement.

### Exercice 4 — Requêtes SQL depuis Python  ★★☆

1. Ouvrez une connexion `sqlite3` à `donnees/assurance.db` et listez les tables
   (`SELECT name FROM sqlite_master WHERE type='table'`).
2. Avec `pd.read_sql`, récupérez les contrats d'un produit passé en **paramètre**
   (jamais de f-string dans du SQL : injection, guillemets…).
3. Calculez le nombre de contrats par produit **en SQL**, puis **en pandas** : même résultat ?

In [ ]:
import sqlite3

cnx = sqlite3.connect(DONNEES / "assurance.db")
# À vous

### Vertica avec VerticaPy

Les cellules suivantes s'exécutent **seulement si une connexion Vertica est configurée** ;
sinon elles l'indiquent et ne font rien (le reste du notebook fonctionne sans Vertica).

- **au datalab** : renseignez `NOM_CONNEXION` (le nom de la connexion verticapy) et `SCHEMA`
  (un schéma où vous avez le droit de créer une table) ;
- **ailleurs** : Vertica local dans Docker (dossier `vertica_local/` du formateur) et les
  variables d'environnement `VERTICA_HOST`, `VERTICA_PORT`, `VERTICA_USER`, `VERTICA_PASSWORD`, `VERTICA_DB`.

Pièges VerticaPy : les conditions et agrégats s'écrivent **en SQL** (chaînes) ; `shape()` et `dtypes()`
sont des méthodes ; `filter` et `sort` modifient le vDataFrame **en place** (`search` renvoie une copie) ;
`to_pandas()` renvoie les `NUMERIC` en `Decimal` (colonne `object`) et les dates en `datetime.date`.

In [ ]:
import os

NOM_CONNEXION = ""                                   # au datalab : nom de la connexion verticapy
SCHEMA = os.environ.get("VERTICA_SCHEMA", "acm")     # au datalab : votre schéma de test

VERTICA = False
try:
    import vertica_python
    import verticapy as vp
except ImportError:
    print("verticapy n'est pas installé : partie Vertica ignorée")
else:
    if NOM_CONNEXION:
        vp.connect(NOM_CONNEXION)
        VERTICA = True
    elif os.environ.get("VERTICA_HOST"):
        # identifiants lus dans l'environnement, jamais écrits dans le notebook
        vp.set_connection(vertica_python.connect(
            host=os.environ["VERTICA_HOST"],
            port=int(os.environ.get("VERTICA_PORT", "5433")),
            user=os.environ.get("VERTICA_USER", "dbadmin"),
            password=os.environ.get("VERTICA_PASSWORD", ""),
            database=os.environ.get("VERTICA_DB", "acm"),
        ))
        VERTICA = True
    else:
        print("Pas de connexion Vertica configurée : les cellules Vertica ne feront rien.")
if VERTICA:
    print("Vertica", vp.vertica_version(), "| verticapy", vp.__version__)

### Exercice 5 — VerticaPy : calculer côté serveur  ★★☆

1. Créez un vDataFrame sur la table `contrats` du schéma `SCHEMA`. Si elle n'existe pas
   (au datalab), créez-la à partir du DataFrame `contrats` avec `vp.read_pandas`.
   Affichez sa taille (`shape()`) et ses types (`dtypes()`).
2. Activez `vp.set_option("sql_on", True)`, puis calculez **dans Vertica** le nombre de contrats et
   la prime totale par produit (`groupby(colonnes, [expressions SQL])`). Quand la requête part-elle ?
3. Rapatriez le résultat avec `to_pandas()` et comparez-le au même calcul fait en pandas
   sur `contrats`. Attention au type de la colonne des montants.
4. ★★★ Listez les primes négatives côté serveur (`search`) et comparez leur nombre avec pandas.

In [ ]:
if VERTICA:
    contrats_v = ...   # À vous
else:
    print("Pas de Vertica : exercice à faire au datalab")

### Exercice 6 — Sélectionner, filtrer, trier  ★☆☆

Sur `contrats` :
1. les contrats `AUTO` actifs dont la prime dépasse 1000 €, triés par prime décroissante
   (colonnes `num_contrat`, `agence`, `prime_annuelle` seulement) ;
2. ajoutez une colonne `duree_jours` = résiliation − effet (en jours) ;
3. combien de contrats ont une durée négative ?

In [ ]:
# À vous

## De SAS à pandas

| SAS | pandas |
|---|---|
| `PROC SORT; BY x;` | `df.sort_values("x")` |
| `PROC SORT NODUPKEY; BY x y;` | `df.drop_duplicates(["x", "y"])` |
| `PROC FREQ; TABLES x;` | `df["x"].value_counts(dropna=False)` |
| `PROC FREQ; TABLES x*y;` | `pd.crosstab(df["x"], df["y"])` |
| `PROC MEANS; CLASS x; VAR v;` | `df.groupby("x")["v"].agg(["count", "mean", "sum"])` |
| `PROC SQL; ... LEFT JOIN ...` | `df1.merge(df2, on="cle", how="left")` |
| `DATA; IF c THEN v = 1; ELSE v = 0;` | `df["v"] = np.where(c, 1, 0)` |
| `WHERE` | `df.loc[condition]` ou `df.query("...")` |
| `%MACRO m(p); ... %MEND;` | `def m(p): ...` |
| `PROC EXPORT` | `df.to_csv(...)` / `df.to_excel(...)` |

L'aide-mémoire complet est dans `aide_memoire_sas_pandas.md`.

### Exercice 7 — Traduire des PROC SAS  ★★☆

Traduisez en pandas :
```sas
PROC FREQ DATA=contrats; TABLES produit*statut / MISSING; RUN;
PROC MEANS DATA=contrats N MEAN SUM; CLASS produit; VAR prime_annuelle; RUN;
PROC SORT DATA=clients OUT=clients_uniques NODUPKEY; BY id_client; RUN;
DATA contrats; SET contrats;
  IF prime_annuelle >= 1000 THEN tranche = "haute"; ELSE tranche = "standard";
RUN;
```

In [ ]:
import numpy as np

# À vous

### Exercice 8 — Premiers contrôles qualité  ★★☆

1. Lignes strictement dupliquées dans `clients` et dans `sinistres` (`duplicated`).
2. Taux de valeurs manquantes par colonne de chaque table (`isna().mean()`), en %.
3. Doublons **de personnes** : même nom, même prénom (sans tenir compte de la casse)
   et même date de naissance, mais identifiants différents.
4. Contrats orphelins (client inexistant) et sinistres sur un contrat inexistant (`isin`).

In [ ]:
# À vous

### Exercice 9 — Exporter les anomalies  ★★★

Rassemblez les contrats orphelins et les contrats à prime négative dans un seul
DataFrame avec une colonne `controle` qui indique la règle violée (`pd.concat`),
puis exportez-le dans `sortie/anomalies_contrats.csv` (UTF-8, `;`).
Créez le dossier avec `pathlib` s'il n'existe pas.

In [ ]:
SORTIE = Path("sortie")
# À vous